# Photoacoustic reconstruction: CUDA smoke on Google Colab

Before running: Runtime > Change runtime type > select a GPU. Then Runtime > Run all.
The notebook records whichever GPU it is given.

What this notebook does, in order: inspect the GPU, check out a pinned commit, install the pinned
dependencies without replacing the platform's CUDA PyTorch, run the tests, run the CUDA smoke
(`scripts/cuda_smoke.py`), validate its verdict, estimate the full-run time and package the results.
With `RUN_FULL = False` it stops there.

Only the PyTorch U-Net uses the GPU. The j-Wave/JAX physics is pinned to CPU (`JAX_PLATFORMS=cpu`).
All scientific code lives in the repository; this notebook only launches it. See `REMOTE_GPU.md`.


## 1. Configuration

In [ ]:
import os
import pathlib
import time

REPO_URL = "https://github.com/AdebanjiAdelowo/photoacoustic-reconstruction.git"
# REF is the full SHA of the commit that is checked out and run. It must contain the remote
# workflow (scripts/cuda_smoke.py); change it in scripts/build_remote_notebooks.py, not here.
REF = "0909a7234abca4f13698d256dc6f5a3814ec0c8e"
RUN_FULL = False  # True runs the gated five-seed CPU-versus-CUDA experiment after the smoke

PLATFORM = "colab"
WORK_DIR = pathlib.Path("/content")
REPO_DIR = WORK_DIR / "photoacoustic-reconstruction"
RESULTS_ROOT = REPO_DIR / "results"

os.environ["JAX_PLATFORMS"] = "cpu"                       # the physics stays on CPU
os.environ["PHOTOACOUSTIC_PROTECT_HISTORICAL"] = "1"      # no writes to report/, data/, experiments/
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"         # deterministic cuBLAS

# One RUN_ID per session: created once, kept in a file, reused when this cell is run again.
_run_id_file = WORK_DIR / "photoacoustic_run_id.txt"
if _run_id_file.exists():
    RUN_ID = _run_id_file.read_text().strip()
else:
    RUN_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "-" + REF[:7] + "-" + PLATFORM
    _run_id_file.write_text(RUN_ID)
RUN_DIR = RESULTS_ROOT / RUN_ID
FULL_GATE_OPEN = RUN_FULL is True
print("RUN_ID", RUN_ID)
print("RUN_DIR", RUN_DIR)
print("RUN_FULL", RUN_FULL)


## 2. GPU and interpreter

In [ ]:
import json
import shutil
import subprocess
import sys


def sh(cmd, cwd=None):
    """Run a command, stream its output here and stop on failure."""
    print("$", " ".join(str(c) for c in cmd))
    proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit status {proc.returncode}: {cmd[0]}")


def py(*args):
    """Run a repository script with this interpreter, from the repository root."""
    sh([sys.executable, *args], cwd=REPO_DIR)


print(sys.version)
sh(["nvidia-smi"])


## 3. Repository at the pinned commit

In [ ]:
if not (REPO_DIR / ".git").exists():
    sh(["git", "clone", REPO_URL, REPO_DIR])
sh(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR)
sh(["git", "checkout", "--detach", REF], cwd=REPO_DIR)

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
assert head == REF, f"HEAD is {head}, expected {REF}"
dirty = subprocess.run(["git", "status", "--porcelain"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout
assert dirty == "", "source tree is not clean:\n" + dirty
print("HEAD", head, "(clean)")


## 4. Dependencies

The platform's CUDA PyTorch is kept; JAX is the CPU build.

In [ ]:
ENV_DIR = RUN_DIR / "env"
py("scripts/remote_setup.py", "preflight", "--out", ENV_DIR / "preflight.json")
py("scripts/remote_setup.py", "install", "--preflight", ENV_DIR / "preflight.json")
py("scripts/remote_setup.py", "verify", "--preflight", ENV_DIR / "preflight.json", "--out", ENV_DIR / "environment.json")


## 5. Tests

In [ ]:
py("-m", "pytest", "-q", "-p", "no:cacheprovider")


## 6. CUDA smoke

In [ ]:
SMOKE_DIR = RUN_DIR / "smoke"
VERDICT = SMOKE_DIR / "smoke_verdict.json"
if VERDICT.exists():
    print("smoke verdict already present for this RUN_ID; not running it again")
else:
    py("scripts/cuda_smoke.py", "--device", "cuda", "--out-dir", SMOKE_DIR)


## 7. Verdict

In [ ]:
py("scripts/cuda_smoke.py", "--validate", VERDICT)

verdict = json.loads(VERDICT.read_text())
assert verdict["passed"] is True and verdict["require_cuda"] is True
assert verdict["provenance"]["resolved_device"].startswith("cuda")
assert verdict["provenance"]["jax_cpu_only"] is True
print("GPU:", verdict["provenance"]["gpu_name"])


## 8. Estimated full-run time

An estimate from the components measured by the smoke, not a measurement.

In [ ]:
py("scripts/remote_experiment.py", "--run-dir", RUN_DIR, "--estimate", VERDICT)


## 9. Package the smoke results

In [ ]:
archive = shutil.make_archive(str(WORK_DIR / f"photoacoustic_{RUN_ID}_smoke"), "zip", root_dir=RESULTS_ROOT, base_dir=RUN_ID)
print("packaged", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:  # not in Colab, or the browser blocked the download
    print(f"download it from the file browser instead: {archive} ({exc})")


## 10. Stop

In [ ]:
if not FULL_GATE_OPEN:
    print("CUDA SMOKE COMPLETE. No full experiment was run.")


## Full experiment (gated)

The cells below do nothing unless `RUN_FULL = True` was set in the configuration cell. With the gate
open they run `scripts/remote_experiment.py`: data generated once on CPU, then the U-Net trained and
evaluated for seeds 0 to 4 on CPU (arm A) and on CUDA (arm B), and the two arms compared. The Tikhonov
and model-mismatch evaluations are not included.


In [ ]:
if FULL_GATE_OPEN:
    py("scripts/remote_experiment.py", "--run-dir", RUN_DIR, "--confirm-full")


In [ ]:
if FULL_GATE_OPEN:
    archive = shutil.make_archive(str(WORK_DIR / f"photoacoustic_{RUN_ID}_full"), "zip", root_dir=RESULTS_ROOT, base_dir=RUN_ID)
    print("packaged", archive)
    print("FULL EXPERIMENT COMPLETE.")
